# 🇨🇦 Project: Canadian Mortgage Credit Risk & Portfolio Analysis
## Business Goal: Identifying Interest Rate Sensitivity
**Objective:** To analyze the current mortgage portfolio for a Canadian bank and determine which borrower segments are most at risk of default given a potential **2% interest rate hike** in the 2026 renewal cycle.

**Key Metrics:**
* **DTI (Debt-to-Income):** Measuring the "squeeze" on household cash flow.
* **Delinquency Rates:** Identifying patterns in historical non-payment.
* **Geographic Concentration:** Comparing risks in Ontario (ON) vs. British Columbia (BC).

### Importing Libraries and Loading Data
Why we do this: We need specialized tools for financial computation. **pandas** is the industry standard for handling tabular data (spreadsheets), and **numpy** is used for fast numerical operations and random simulations.

> Always check your raw data first. Data often comes from different legacy systems and may have "garbage" values that need identifying early.

In [4]:
import pandas as pd
import numpy as np

# Load the raw data
df = pd.read_csv('raw_loan_data.csv')
print("Initial Load Successful. Dataframe shape:", df.shape)

# Look at the first 5 rows and all column names
print("Column Names in this dataset:")
print(df.columns.tolist())
df.head()

Initial Load Successful. Dataframe shape: (50000, 20)
Column Names in this dataset:
['customer_id', 'age', 'occupation_status', 'years_employed', 'annual_income', 'credit_score', 'credit_history_years', 'savings_assets', 'current_debt', 'defaults_on_file', 'delinquencies_last_2yrs', 'derogatory_marks', 'product_type', 'loan_intent', 'loan_amount', 'interest_rate', 'debt_to_income_ratio', 'loan_to_income_ratio', 'payment_to_income_ratio', 'loan_status']


,customer_id,age,occupation_status,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,product_type,loan_intent,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,loan_status
0,CUST100000,40,Employed,17.2,25579,692,5.3,895,10820,0,0,0,Credit Card,Business,600,17.02,0.423,0.023,0.008,1
1,CUST100001,33,Employed,7.3,43087,627,3.5,169,16550,0,1,0,Personal Loan,Home Improvement,53300,14.10,0.384,1.237,0.412,0
2,CUST100002,42,Student,1.1,20840,689,8.4,17,7852,0,0,0,Credit Card,Debt Consolidation,2100,18.33,0.377,0.101,0.034,1
3,CUST100003,53,Student,0.5,29147,692,9.8,1480,11603,0,1,0,Credit Card,Business,2900,18.74,0.398,0.099,0.033,1
4,CUST100004,32,Employed,12.5,63657,630,7.2,209,12424,0,0,0,Personal Loan,Education,99600,13.92,0.195,1.565,0.522,1


### 1. Regional Segmentation (Adding Provinces)
Why we do this: In the Canadian market, risk is not uniform. Real estate in Ontario and BC carries different risk profiles due to varying house prices and debt levels. 
We use **np.random.choice** to simulate this regional data so we can later perform **Cohort Analysis** (comparing one group to another).
* **Goal:** Create provincial cohorts to compare default sensitivity.
* **Logic:** Using a 60/40 weighted distribution to reflect realistic loan volumes.

>Data Simulation is a vital skill when you need to test a theory but don't have the specific variable yet. Proving you can "enrich" a dataset shows you think about the business context.

In [6]:
# We use p=[0.6, 0.4] to reflect that Ontario has a larger population/loan volume
df['province'] = np.random.choice(['Ontario', 'British Columbia'], size=len(df), p=[0.6, 0.4])

# Verify the distribution
print(df['province'].value_counts(normalize=True))

# Preview the new column
df[['customer_id', 'province']].head()

province
Ontario             0.59828
British Columbia    0.40172
Name: proportion, dtype: float64


,customer_id,province
0,CUST100000,Ontario
1,CUST100001,Ontario
2,CUST100002,Ontario
3,CUST100003,Ontario
4,CUST100004,Ontario


### 2. Normalizing Metrics: Monthly Income & DTI
Why we do this: We normalize annual income to identify the "Monthly Squeeze."
"Annual Income" is a big number, but banks care about Monthly Cash Flow. We need to create a monthly_income column to calculate accurate Debt-to-Income (DTI) trends, which is one of your primary project goals.
> This is Feature Transformation : Transforming raw debt data into actionable percentages (DTI) ensures our report speaks the language of bank stakeholders.

In [7]:
# Normalizing annual data to monthly for standard banking reports
df['monthly_income'] = df['annual_income'] / 12

# In this dataset, 'debt_to_income_ratio' is already provided, 
# but we normalize it to a 0-100 scale if it's currently a decimal
if df['debt_to_income_ratio'].max() <= 1.0:
    df['debt_to_income_ratio'] = df['debt_to_income_ratio'] * 100

print(f"Average DTI across portfolio: {df['debt_to_income_ratio'].mean():.2f}%")

Average DTI across portfolio: 28.57%


### 3. The 2026 "Renewal Cliff" Stress Test
Why we do this: This is our most critical insight. We are creating a hypothetical scenario where interest rates rise by 2%. This allows us to identify which customers are currently "Safe" but would become "High Risk" (DTI > 40%) if their mortgage renewed today.  

> This helps the bank forecast potential credit losses before they happen. Stress Testing evaluates financial resilience under hypothetical negative events.


In [8]:
# Simulate a 2% interest rate hike
df['stressed_interest_rate'] = df['interest_rate'] + 2.0

# Identify 'At-Risk' customers (High DTI + Rising Rates)
at_risk_count = df[(df['debt_to_income_ratio'] > 40) & (df['interest_rate'] > 5)].shape[0]
print(f"Number of customers at high risk in a rising rate environment: {at_risk_count}")

Number of customers at high risk in a rising rate environment: 11564


### 4. Data Pipeline: Exporting Cleaned Asset
Why we do this: Python handles the math; **SQL** and **Power BI** will handle the final executive reporting. We are saving this "Stressed" dataset as our single source of truth for the dashboard.Exporting Clean Data for Power BI/SQL

> This represents the Data Pipeline. You’ve taken raw, generic data and turned it into a "Canadianized" asset ready for executive-level dashboards.


In [10]:
# Save the final version
df.to_csv('canadian_loan_data_clean.csv', index=False)